In [ ]:
# Core libs for file handling, arrays, dataframes, and plotting
import os  # filesystem paths and directory utilities
import numpy as np  # vectorized numerical ops
import pandas as pd  # tabular data handling
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import confusion_matrix, classification_report  # evaluation metrics

from PIL import Image  # image I/O and basic manipulation

# Deep learning stack (TensorFlow/Keras) and computer vision helpers
import tensorflow as tf  # core TensorFlow engine
from tensorflow import keras  # Keras high-level API
from tensorflow.keras.models import Sequential  # linear stack of layers
from tensorflow.keras.optimizers import Adam, Adamax  # optimizers
from tensorflow.keras.metrics import categorical_crossentropy  # loss metric alias
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # augmentation pipeline
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization  # layer building blocks
from tensorflow.keras import regularizers  # weight regularization
import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

from skimage.measure import shannon_entropy

from collections import Counter

from sklearn.decomposition import PCA

import random
import shutil

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

In [ ]:
source_dir = "dataset"
dest_dir = "dataset/colon_dataset"

classes = ["colon_aca", "colon_n"]

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

for cls in classes:

    class_path = os.path.join(source_dir, cls)
    images = os.listdir(class_path)

    random.shuffle(images)

    train_end = int(len(images)*train_ratio)
    val_end = int(len(images)*(train_ratio+val_ratio))

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    splits = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }

    for split, img_list in splits.items():

        dest_folder = os.path.join(dest_dir, split, cls)
        os.makedirs(dest_folder, exist_ok=True)

        for img in img_list:

            shutil.move(
                os.path.join(class_path, img),
                os.path.join(dest_folder, img)
            )

print("Colon dataset successfully split.")

In [ ]:
for split in ["train","val","test"]:
    for cls in ["colon_aca","colon_n"]:
        path = os.path.join("dataset/colon_dataset",split,cls)
        print(split, cls, len(os.listdir(path)))

In [ ]:
base = "dataset/colon_dataset"

data = []

for split in ["train","val","test"]:
    for cls in ["colon_aca","colon_n"]:
        path = os.path.join(base,split,cls)
        data.append([split,cls,len(os.listdir(path))])

df = pd.DataFrame(data,columns=["Split","Class","Count"])

sns.barplot(data=df,x="Split",y="Count",hue="Class")

plt.title("Colon Dataset Class Distribution")
plt.show()

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(10,6))

splits=["train","val","test"]

for i,split in enumerate(splits):

    for j,cls in enumerate(["colon_aca","colon_n"]):

        folder=os.path.join(base,split,cls)
        img_path=os.path.join(folder,random.choice(os.listdir(folder)))

        img=cv2.imread(img_path)
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        ax[j,i].imshow(img)
        ax[j,i].set_title(f"{cls}-{split}")
        ax[j,i].axis("off")

plt.show()

In [ ]:
width=[]
height=[]

for split in ["train","val","test"]:
    for cls in ["colon_aca","colon_n"]:

        folder=os.path.join(base,split,cls)

        for img_name in os.listdir(folder)[:200]:

            img=Image.open(os.path.join(folder,img_name))
            w,h=img.size

            width.append(w)
            height.append(h)

plt.scatter(width,height)

plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Size Distribution")
plt.show()

In [ ]:
brightness=[]

for split in ["train","val","test"]:
    for cls in ["colon_aca","colon_n"]:

        folder=os.path.join(base,split,cls)

        for img_name in os.listdir(folder)[:200]:

            img=cv2.imread(os.path.join(folder,img_name))
            brightness.append(np.mean(img))

sns.histplot(brightness,bins=30)

plt.title("Brightness Distribution")
plt.show()